conda create -n ai-econ-legacy python=3.7.16 -y
conda activate ai-econ-legacy

pip install "numpy==1.16.4"
pip install "tensorflow==1.14.0"
pip install "gym<0.23"
pip install "ray[rllib]==1.13.0"
pip install jupyter ipykernel tensorboard
python -m ipykernel install --user --name ai-econ-legacy --display-name "Python (ai-econ-legacy)"

In [1]:
import os, signal, sys, time

sys.path.append(r'C:\Users\adria\coding\katja\DRL-in-international-economy-ai-economist-')

from ai_economist import foundation

import numpy as np

from utils import plotting

import ray
from ray.rllib.agents.ppo import PPOTrainer

c:\Users\adria\anaconda3\envs\ai-economist\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
c:\Users\adria\anaconda3\envs\ai-economist\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
c:\Users\adria\anaconda3\envs\ai-economist\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
c:\Users\adria\anaconda3\envs\ai-economist\lib\site-

In [2]:
from rllib.env_wrapper import RLlibEnvWrapper


In [ ]:
import sys
import ray

print("Python:", sys.version)
print("Ray:", ray.__version__)

Python: 3.7.16 (default, Jan 17 2023, 16:06:28) [MSC v.1916 64 bit (AMD64)]
Ray: 0.8.6


: 

In [3]:
import sys
import os
import ray
import numpy as np
import tensorflow as tf
import gym

print("Python:", sys.version)
print("Ray:", ray.__version__)
print("NumPy:", np.__version__)
print("TensorFlow:", tf.__version__)
print("Gym:", gym.__version__)

Python: 3.7.16 (default, Jan 17 2023, 16:06:28) [MSC v.1916 64 bit (AMD64)]
Ray: 0.8.6
NumPy: 1.21.0
TensorFlow: 1.14.0
Gym: 0.26.2


In [ ]:
import os
import time
import gc
import ray

from ray.rllib.agents.ppo import PPOTrainer
from ray.tune.logger import UnifiedLogger
from tutorials.rllib.env_wrapper import RLlibEnvWrapper

# =========================================================
# Init Ray once
# =========================================================
ray.init(ignore_reinit_error=True, log_to_driver=False)

PHASE1_DIR = "phase1_tb"
PHASE2_DIR = "phase2_tb"
os.makedirs(PHASE1_DIR, exist_ok=True)
os.makedirs(PHASE2_DIR, exist_ok=True)

def make_logger_creator(base_dir):
    def logger_creator(config):
        ts = time.strftime("%Y-%m-%d_%H-%M-%S")
        logdir = os.path.join(base_dir, f"run-{ts}")
        os.makedirs(logdir, exist_ok=True)
        return UnifiedLogger(config, logdir, loggers=None)
    return logger_creator

def policy_mapping_fun(agent_id):
    aid = str(agent_id)
    if aid.isdigit():
        return "a"
    if aid == "p_top":
        return "p_top"
    if aid == "p_bottom":
        return "p_bottom"
    return "a"

# =========================================================
# Phase 1 env
# =========================================================
env_config_dict_phase1 = {
    "scenario_name": "custom/splitworld_overlay_regional",
    "components": [
        ('Build', {'skill_dist':'pareto','payment_max_skill_multiplier':3,'build_labor':10,'payment':10}),
        ('ContinuousDoubleAuction', {'max_bid_ask':10,'order_labor':0.25,'max_num_orders':5,'order_duration':50}),
        ('Gather', {'move_labor':1,'collect_labor':1,'skill_dist':'pareto'}),
        ("RegionalPeriodicBracketTax", {
            "region":"top","planner_id":"p_top","period":100,"bracket_spacing":"us-federal","usd_scaling":1000,
            "disable_taxes": True,
        }),
        ("RegionalPeriodicBracketTax", {
            "region":"bottom","planner_id":"p_bottom","period":100,"bracket_spacing":"us-federal","usd_scaling":1000,
            "disable_taxes": True,
        }),
    ],
    "env_layout_file":"map_100x50_water_gaps_3percent_resources.txt",
    "world_size":[100,50],
    "episode_length":1000,
    "starting_agent_coin":10,
    "fixed_four_skill_and_loc":False,
    "n_agents":4,
    "planner_subclasses":["TopPlanner","BottomPlanner"],
    "multi_action_mode_planner":True,
    "multi_action_mode_agents":True,
    "flatten_observations":True,
    "flatten_masks":True,
    "dense_log_frequency":1
}

# Build once to capture spaces
env_obj_phase1 = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase1}, verbose=False)

obs_space_a   = env_obj_phase1.observation_space
act_space_a   = env_obj_phase1.action_space
obs_space_top = env_obj_phase1.observation_space_pl["p_top"]
act_space_top = env_obj_phase1.action_space_pl["p_top"]
obs_space_bot = env_obj_phase1.observation_space_pl["p_bottom"]
act_space_bot = env_obj_phase1.action_space_pl["p_bottom"]

policies = {
    "a": (None, obs_space_a, act_space_a, {"lr": 3e-4}),
    "p_top": (None, obs_space_top, act_space_top, {"lr": 1e-4}),
    "p_bottom": (None, obs_space_bot, act_space_bot, {"lr": 1e-4}),
}

def build_trainer_config(env_config_dict, policies_to_train):
    return {
        "env": RLlibEnvWrapper,
        "env_config": {
            "env_config_dict": env_config_dict,
            "num_envs_per_worker": 1,
        },
        "multiagent": {
            "policies": policies,
            "policies_to_train": policies_to_train,
            "policy_mapping_fn": policy_mapping_fun,
        },

        # single-process notebook-safe
        "num_workers": 0,
        "num_envs_per_worker": 1,

        # TF only, since torch is not installed
        "framework": "tf",

        # lower memory pressure
        "rollout_fragment_length": 50,
        "batch_mode": "truncate_episodes",
        "train_batch_size": 500,
        "sgd_minibatch_size": 64,
        "num_sgd_iter": 2,

        # extra safety
        #"num_gpus": 0,
        "log_level": "WARN",
    }

def run_phase(name, env_config_dict, policies_to_train, log_dir, iters, restore_path=None):
    trainer = PPOTrainer(
        config=build_trainer_config(env_config_dict, policies_to_train),
        logger_creator=make_logger_creator(log_dir),
    )

    if restore_path is not None:
        trainer.restore(restore_path)
        print(f"[{name}] Restored from {restore_path}")

    ckpt_path = None
    for i in range(iters):
        result = trainer.train()

        if i % 25 == 0:
            print(f"[{name}] Iter={i:05d} reward={result.get('episode_reward_mean')}")

        if i % 100 == 0 and i > 0:
            ckpt_path = trainer.save(log_dir)
            print(f"[{name}] Saved: {ckpt_path}")

        if i % 10 == 0:
            gc.collect()

    ckpt_path = trainer.save(log_dir)
    print(f"[{name}] Final checkpoint: {ckpt_path}")

    trainer.stop()
    del trainer
    gc.collect()
    return ckpt_path

# =========================================================
# Run Phase 1
# =========================================================
ckpt_phase1_path = run_phase(
    name="PHASE 1",
    env_config_dict=env_config_dict_phase1,
    policies_to_train=["a"],
    log_dir=PHASE1_DIR,
    iters=2000,
)

# =========================================================
# Phase 2 env
# =========================================================
env_config_dict_phase2 = {
    "scenario_name": "custom/splitworld_overlay_regional",
    "components": [
        ('Build', {'skill_dist':'pareto','payment_max_skill_multiplier':3,'build_labor':10,'payment':10}),
        ('ContinuousDoubleAuction', {'max_bid_ask':10,'order_labor':0.25,'max_num_orders':5,'order_duration':50}),
        ('Gather', {'move_labor':1,'collect_labor':1,'skill_dist':'pareto'}),
        ("RegionalPeriodicBracketTax", {
            "region":"top","planner_id":"p_top","period":100,"bracket_spacing":"us-federal","usd_scaling":1000,
            "disable_taxes": False, "tax_model":"model_wrapper", "tax_annealing_schedule":[-100, 0.001]
        }),
        ("RegionalPeriodicBracketTax", {
            "region":"bottom","planner_id":"p_bottom","period":100,"bracket_spacing":"us-federal","usd_scaling":1000,
            "disable_taxes": False, "tax_model":"model_wrapper", "tax_annealing_schedule":[-100, 0.001]
        }),
    ],
    "env_layout_file":"map_100x50_water_gaps_3percent_resources.txt",
    "world_size":[100,50],
    "episode_length":1000,
    "starting_agent_coin":10,
    "fixed_four_skill_and_loc":False,
    "n_agents":4,
    "planner_subclasses":["TopPlanner","BottomPlanner"],
    "multi_action_mode_planner":True,
    "multi_action_mode_agents":True,
    "flatten_observations":True,
    "flatten_masks":True,
    "dense_log_frequency":1
}

# =========================================================
# Run Phase 2
# =========================================================
ckpt_phase2_path = run_phase(
    name="PHASE 2",
    env_config_dict=env_config_dict_phase2,
    policies_to_train=["p_top", "p_bottom"],
    log_dir=PHASE2_DIR,
    iters=5000,
    restore_path=ckpt_phase1_path,
)

print("Done.")
print("Phase 1 checkpoint:", ckpt_phase1_path)
print("Phase 2 checkpoint:", ckpt_phase2_path)

ray.shutdown()

2026-03-18 11:48:05,146	INFO resource_spec.py:212 -- Starting Ray with 1.56 GiB memory available for workers and up to 0.8 GiB for objects. You can adjust these settings with ray.init(memory=<bytes>, object_store_memory=<bytes>).
2026-03-18 11:48:05,512	INFO services.py:1165 -- View the Ray dashboard at localhost:8265


In [1]:
import torch
print(torch.__version__)

ModuleNotFoundError: No module named 'torch'

In [2]:
import sys, ray

print("Python:", sys.version)
print("Ray:", ray.__version__)

try:
    from ray.rllib.agents.ppo import PPOTrainer
    print("Old RLlib API available: PPOTrainer")
except Exception as e:
    print("Old API import failed:", repr(e))

try:
    from ray.rllib.algorithms.ppo import PPOConfig
    print("New RLlib API available: PPOConfig")
except Exception as e:
    print("New API import failed:", repr(e))

try:
    import torch
    print("Torch:", torch.__version__)
except Exception as e:
    print("Torch import failed:", repr(e))

try:
    import tensorflow as tf
    print("TensorFlow:", tf.__version__)
except Exception as e:
    print("TensorFlow import failed:", repr(e))

Python: 3.7.16 (default, Jan 17 2023, 16:06:28) [MSC v.1916 64 bit (AMD64)]
Ray: 0.8.6


c:\Users\adria\anaconda3\envs\ai-economist\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
c:\Users\adria\anaconda3\envs\ai-economist\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
c:\Users\adria\anaconda3\envs\ai-economist\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
c:\Users\adria\anaconda3\envs\ai-economist\lib\site-

Old RLlib API available: PPOTrainer
New API import failed: ModuleNotFoundError("No module named 'ray.rllib.algorithms'")
Torch import failed: ModuleNotFoundError("No module named 'torch'")
TensorFlow: 1.14.0


In [ ]:
import os
from ray.rllib.agents.ppo import PPOTrainer
from ray.tune.logger import UnifiedLogger
from tutorials.rllib.env_wrapper import RLlibEnvWrapper
import time


import ray
ray.init(ignore_reinit_error=True, log_to_driver=False)  # Dashboard not needed for TB

# ----------------------------
# Output dirs (TensorBoard + ckpts)
# ----------------------------
PHASE1_DIR = "phase1_tb"
PHASE2_DIR = "phase2_tb"
os.makedirs(PHASE1_DIR, exist_ok=True)
os.makedirs(PHASE2_DIR, exist_ok=True)

def make_logger_creator(base_dir):
    def logger_creator(config):
        ts = time.strftime("%Y-%m-%d_%H-%M-%S")
        logdir = os.path.join(base_dir, f"run-{ts}")
        os.makedirs(logdir, exist_ok=True)
        return UnifiedLogger(config, logdir, loggers=None)
    return logger_creator

# ==========================================
# PHASE 1 — ENV (agents-only; taxes disabled)
# ==========================================
env_config_dict_phase1 = {
    "scenario_name": "custom/splitworld_overlay_regional",
    "components": [
        ('Build', {'skill_dist':'pareto','payment_max_skill_multiplier':3,'build_labor':10,'payment':10}),

        ('ContinuousDoubleAuction', {'max_bid_ask':10,'order_labor':0.25,'max_num_orders':5,'order_duration':50}),

        ('Gather', {'move_labor':1,'collect_labor':1,'skill_dist':'pareto'}),

        ("RegionalPeriodicBracketTax", {
            "region":"top","planner_id":"p_top","period":100,"bracket_spacing":"us-federal","usd_scaling":1000,
            "disable_taxes": True,
        }),

        ("RegionalPeriodicBracketTax", {
            "region":"bottom","planner_id":"p_bottom","period":100,"bracket_spacing":"us-federal","usd_scaling":1000,
            "disable_taxes": True,
        }),
    ],
    "env_layout_file":"map_100x50_water_gaps_3percent_resources.txt",
    "world_size":[100,50],
    "episode_length":1000,
    "starting_agent_coin":10,
    "fixed_four_skill_and_loc":False,
    "n_agents":4,
    "planner_subclasses":["TopPlanner","BottomPlanner"],
    "multi_action_mode_planner":True,
    "multi_action_mode_agents":True,
    "flatten_observations":True,
    "flatten_masks":True,
    "dense_log_frequency":1
}

# Build a wrapper once to capture spaces for policies
env_obj_phase1 = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase1}, verbose=False)

# Per-policy learning rates
obs_space_a   = env_obj_phase1.observation_space
act_space_a   = env_obj_phase1.action_space
obs_space_top = env_obj_phase1.observation_space_pl["p_top"]
act_space_top = env_obj_phase1.action_space_pl["p_top"]
obs_space_bot = env_obj_phase1.observation_space_pl["p_bottom"]
act_space_bot = env_obj_phase1.action_space_pl["p_bottom"]

policies = {
    "a": (
        None, obs_space_a, act_space_a,
        {"lr": 3e-4}   # agents (Phase 1)
    ),
    "p_top": (
        None, obs_space_top, act_space_top,
        {"lr": 1e-4}   # planners (Phase 2)
    ),
    "p_bottom": (
        None, obs_space_bot, act_space_bot,
        {"lr": 1e-4}
    ),
}

def policy_mapping_fun(agent_id):
    aid = str(agent_id)
    if aid.isdigit():
        return "a"
    if aid == "p_top":
        return "p_top"
    if aid == "p_bottom":
        return "p_bottom"
    return "a"

trainer_config_phase1 = {
    "env": RLlibEnvWrapper,
    "env_config": {
        "env_config_dict": env_config_dict_phase1,
        "num_envs_per_worker": 1,     # <-- REQUIRED by your env_wrapper
    },
    "multiagent": {
        "policies": policies,
        "policies_to_train": ["a"],   # agents only
        "policy_mapping_fn": policy_mapping_fun,
    },
    "num_workers": 0,
    "num_envs_per_worker": 1,
    "framework": "tf",
    "train_batch_size": 1000, # prev 4000
    "sgd_minibatch_size": 256, # prev 4000
    "num_sgd_iter": 1,
}

# Use logger_creator for TensorBoard under PHASE1_DIR
trainer_phase1 = PPOTrainer(
    config=trainer_config_phase1,
    logger_creator=make_logger_creator(PHASE1_DIR)
)

# ---- Phase 1 loop ----
PHASE1_ITERS = 2000
ckpt_phase1_path = None
for i in range(PHASE1_ITERS):
    result = trainer_phase1.train()
    if i % 50 == 0:
        print(f"[PHASE 1] Iter={i:05d} reward={result.get('episode_reward_mean')}")
    if i % 200 == 0 and i > 0:
        ckpt_phase1_path = trainer_phase1.save(PHASE1_DIR)
        print(f"[PHASE 1] Saved: {ckpt_phase1_path}")

ckpt_phase1_path = trainer_phase1.save(PHASE1_DIR)
print(f"[PHASE 1] Final checkpoint: {ckpt_phase1_path}")

# After saving Phase-1 final checkpoint:
trainer_phase1.stop()    # <-- Free Phase-1 workers and resources
ray.shutdown()           # <-- Reset Ray runtime
ray.init(ignore_reinit_error=True, log_to_driver=False)  # <-- Start fresh for Phase-2
# ==========================================
# PHASE 2 — ENV (planners-only; taxes ON)
# ==========================================
env_config_dict_phase2 = {
    "scenario_name": "custom/splitworld_overlay_regional",
    "components": [
        ('Build', {'skill_dist':'pareto','payment_max_skill_multiplier':3,'build_labor':10,'payment':10}),

        ('ContinuousDoubleAuction', {'max_bid_ask':10,'order_labor':0.25,'max_num_orders':5,'order_duration':50}),

        ('Gather', {'move_labor':1,'collect_labor':1,'skill_dist':'pareto'}),

        ("RegionalPeriodicBracketTax", {
            "region":"top","planner_id":"p_top","period":100,"bracket_spacing":"us-federal","usd_scaling":1000,
            "disable_taxes": False, "tax_model":"model_wrapper", "tax_annealing_schedule":[-100, 0.001]
        }),

        ("RegionalPeriodicBracketTax", {
            "region":"bottom","planner_id":"p_bottom","period":100,"bracket_spacing":"us-federal","usd_scaling":1000,
            "disable_taxes": False, "tax_model":"model_wrapper", "tax_annealing_schedule":[-100, 0.001]
        }),
    ],
    "env_layout_file":"map_100x50_water_gaps_3percent_resources.txt",
    "world_size":[100,50],
    "episode_length":1000,
    "starting_agent_coin":10,
    "fixed_four_skill_and_loc":False,
    "n_agents":4,
    "planner_subclasses":["TopPlanner","BottomPlanner"],
    "multi_action_mode_planner":True,
    "multi_action_mode_agents":True,
    "flatten_observations":True,
    "flatten_masks":True,
    "dense_log_frequency":1
}

env_obj_phase2 = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase2}, verbose=False)

trainer_config_phase2 = {
    "env": RLlibEnvWrapper,
    "env_config": {
        "env_config_dict": env_config_dict_phase2,
        "num_envs_per_worker": 1,     # <-- REQUIRED by your env_wrapper
    },
    "multiagent": {
        "policies": policies,          # same policies (per-policy LR applies)
        "policies_to_train": ["p_top", "p_bottom"],
        "policy_mapping_fn": policy_mapping_fun,
    },
    "num_workers": 0,
    "num_envs_per_worker": 1,
    "framework": "tf",
    "train_batch_size": 4000,
    "sgd_minibatch_size": 4000,
    "num_sgd_iter": 1,
}

trainer_phase2 = PPOTrainer(
    config=trainer_config_phase2,
    logger_creator=make_logger_creator(PHASE2_DIR)   # TB logs under PHASE2_DIR
)
trainer_phase2.restore(ckpt_phase1_path)
print(f"[PHASE 2] Restored from Phase-1 checkpoint: {ckpt_phase1_path}")

# Verify annealing after restore
for comp in trainer_phase2.workers.local_worker().env.env.components:
    if "BracketTax" in comp.name:
        print("[Verify-Start]", comp.name, "schedule:", comp.tax_annealing_schedule,
              "annealed_max:", getattr(comp, "_annealed_rate_max", None))

PHASE2_ITERS = 5000
ckpt_phase2_path = None
for i in range(PHASE2_ITERS):
    result = trainer_phase2.train()
    if i % 50 == 0:
        print(f"[PHASE 2] Iter={i:05d} reward={result.get('episode_reward_mean')}")

    if i % 500 == 0 and i > 0:
        for comp in trainer_phase2.workers.local_worker().env.env.components:
            if "BracketTax" in comp.name:
                print("[Verify-Loop]", comp.name,
                      "annealed_max:", getattr(comp, "_annealed_rate_max", None))

    if i % 200 == 0 and i > 0:
        ckpt_phase2_path = trainer_phase2.save(PHASE2_DIR)
        print(f"[PHASE 2] Saved: {ckpt_phase2_path}")

ckpt_phase2_path = trainer_phase2.save(PHASE2_DIR)
print(f"[PHASE 2] Final checkpoint: {ckpt_phase2_path}")


2026-03-18 11:31:16,126	INFO resource_spec.py:212 -- Starting Ray with 2.0 GiB memory available for workers and up to 1.02 GiB for objects. You can adjust these settings with ray.init(memory=<bytes>, object_store_memory=<bytes>).
2026-03-18 11:31:16,628	INFO services.py:1165 -- View the Ray dashboard at localhost:8265


tensorboard --logdir ./phase1_tb
tensorboard --logdir ./phase2_tb

http://localhost:6006

What the official scripts do (and how your notebook compares)
1) Runner & Logging

Official: python training_script.py --run-dir phase1 / phase2

Uses a YAML config, TensorFlow/Keras models, Ray Tune style logging to ~/ray_results, plus two folders per run:

phaseX/ckpts with policy checkpoints
phaseX/dense_logs (compressed .lz4), written by the script




Your notebook: Directly constructs PPOTrainer (no Tune), with a logger_creator so TensorBoard logs go to ./phase1_tb and ./phase2_tb instead of ~/ray_results.
✅ This is fine. It’s equivalent logging, just in a location you control.
📌 If you want to log under ~/ray_results like the docs say, remove the logger_creator and use Tune (tune.run) or pass a logger_creator pointing there.

2) Framework

Official: TensorFlow/Keras models (their examples say “restore_tf_weights_agents”).
Your notebook: We switched to framework: "tf" to avoid the Torch import error.
✅ This matches the official setup better and is stable.

3) Phase 1 vs Phase 2 who trains

Official:

Phase 1: Agents train, planner disabled.
Phase 2: Agents continue training and planner trains (both update), and agents are initialized from a TF checkpoint (restore_tf_weights_agents).


Your notebook (current config I provided):

Phase 1: Agents train (✅).
Phase 2: Planners train only (agents frozen) — that’s an intentional simplification to stabilize the planner faster.
The restore we do is full trainer checkpoint restore (not a TF‑weights‑only restore). This is typically easier in notebooks.



If you want to match the official behavior more closely in Phase 2 (agents continue learning):
Python

- Phase-2: train both agents and planners
- trainer_config_phase2["multiagent"]["policies_to_train"] = ["a", "p_top", "p_bottom"]
- You can also set the agent LR lower during phase-2 (optional):
- policies["a"][3]["lr"] = 1e-5  # e.g., smaller than in phase-1Show more lines
  
4) Annealing

Official: They pass tax_annealing_schedule via YAML.
Your notebook: You now pass "tax_annealing_schedule": [-100, 0.001] in Phase‑2 component configs (✅).
You also added enforcement in generate_masks() to actually bind the annealed max (✅).
When to verify: Run the annealing verification after creating/restoring Phase‑2 trainer and optionally every N iterations.

5) Dense logs

Official: The training script writes dense logs (phaseX/dense_logs/…lz4), which they visualize later.
Your notebook: TensorBoard logs are saved; dense logs are in memory via env.env.dense_log during rollout. If you want files the same way they do:

Either switch to the training script style, or
Periodically (e.g., every 200 iterations) pull the logs from the local worker and save them via the Foundation utils (if you want .lz4):
Python
- from ai_economist.foundation.utils import save_lz4
- env = trainer_phase2.workers.local_worker().env  # RLlib wrapper
- dense_log = env.env.dense_log                    # underlying Foundation env# Pick your path and save:
- save_lz4(dense_log, os.path.join(PHASE2_DIR, f"dense_log_iter_{i}.lz4"))Show more lines

Make sure dense_log_frequency is set (it is, to 1 in your configs).




Does your notebook still work optimally given their guidance?
Yes—for your custom two‑planner environment and for long notebook runs. It is intentionally simpler:

Direct PPOTrainer usage (no CLI) for ease of debugging.
TensorBoard logs go to your chosen folders (./phase1_tb, ./phase2_tb), not ~/ray_results.
Phase‑2 trains only planners by default (more stable early on), but you can easily enable agent training too (see snippet above) to match the tutorial’s description.
Annealing is passed and enforced (good).
Windows‑safe (num_workers = 0) and avoids Torch setup by using TF.

If your goal is maximum parity with the official script:

Use TensorFlow (already doing).
Enable agent training in Phase‑2 (policies_to_train = ["a", "p_top", "p_bottom"]).
Save dense logs to files periodically with save_lz4.
Consider using the training_script.py pattern with YAML if you want the exact folder layout and CLI workflow they document. For notebooks, your approach is excellent.


Minimal changes to match their “Phase‑2: agents continue learning” policy
Add this just before constructing trainer_phase2:
- Train both agents and planners in Phase-2 (closer to official doc)
- trainer_config_phase2["multiagent"]["policies_to_train"] = ["a", "p_top", "p_bottom"]
- Optional: Reduce agent LR during Phase-2 so planners dominate the change
- policies["a"][3]["lr"] = 1e-5   # from 3e-4 in phase-1 to 1e-5 in phase-2

In [ ]:
# =========================
# Two-phase RLlib training
# =========================
import os
import time
import ray

from ray.rllib.agents.ppo import PPOTrainer
from ray.tune.logger import UnifiedLogger
from tutorials.rllib.env_wrapper import RLlibEnvWrapper

# -----------------------------------------------------
# Ray init (required before restore/put, Windows-safe)
# -----------------------------------------------------
ray.init(ignore_reinit_error=True, log_to_driver=False)

# -----------------------------------------------------
# TensorBoard / checkpoint dirs
# -----------------------------------------------------
PHASE1_DIR = "phase1_tb"
PHASE2_DIR = "phase2_tb"
os.makedirs(PHASE1_DIR, exist_ok=True)
os.makedirs(PHASE2_DIR, exist_ok=True)

def make_logger_creator(base_dir):
    """Return a logger_creator that writes RLlib logs to base_dir/run-<timestamp>."""
    def logger_creator(config):
        ts = time.strftime("%Y-%m-%d_%H-%M-%S")
        logdir = os.path.join(base_dir, f"run-{ts}")
        os.makedirs(logdir, exist_ok=True)
        return UnifiedLogger(config, logdir, loggers=None)
    return logger_creator


# -----------------------------------------------------
# Generic training loop (parameterized)
# -----------------------------------------------------
def train_loop(
    trainer,
    *,
    train_iters,
    phase_tag,                    # e.g., "PHASE 1" or "PHASE 2"
    print_every=50,
    save_every=200,
    save_dir=None,
    verify_every=None,            # e.g., 500 for annealing check; None to disable
    verify_fn=None
):
    """
    Generic training loop with flexible iteration counts and modulo-based hooks.
    Ensures:
      - prints at i=0
      - saves at the end (even if not aligned with save_every)
      - optional annealing verify runs every verify_every iterations
    """
    last_ckpt = None
    for i in range(train_iters):
        result = trainer.train()

        # Print: at i=0 and then every print_every (if >0)
        if (i == 0) or (print_every and (i % print_every == 0)):
            print(f"[{phase_tag}] Iter={i:05d} reward={result.get('episode_reward_mean')}")

        # Verify: every verify_every (if provided)
        if verify_every and i > 0 and (i % verify_every == 0) and callable(verify_fn):
            verify_fn(trainer, tag="[Verify-Loop]")

        # Save: every save_every (if provided)
        if save_every and i > 0 and (i % save_every == 0) and save_dir:
            last_ckpt = trainer.save(save_dir)
            print(f"[{phase_tag}] Saved: {last_ckpt}")

    # Final save
    if save_dir:
        last_ckpt = trainer.save(save_dir)
        print(f"[{phase_tag}] Final checkpoint: {last_ckpt}")

    return last_ckpt


# -----------------------------------------------------
# Annealing verification (Phase-2)
# -----------------------------------------------------
def verify_annealing(trainer, tag="[Verify]"):
    env = trainer.workers.local_worker().env.env  # RLlib wrapper -> Foundation env
    for comp in getattr(env, "components", []):
        if "BracketTax" in comp.name:
            print(f"{tag} {comp.name} schedule: {comp.tax_annealing_schedule}, "
                  f"annealed_max: {getattr(comp, '_annealed_rate_max', None)}")


# =====================================================
# PHASE 1 — ENVIRONMENT (agents-only; taxes disabled)
# =====================================================
env_config_dict_phase1 = {
    "scenario_name": "custom/splitworld_overlay_regional",
    "components": [
        ('Build', {
            'skill_dist': 'pareto',
            'payment_max_skill_multiplier': 3,
            'build_labor': 10,
            'payment': 10
        }),
        ('ContinuousDoubleAuction', {
            'max_bid_ask': 10,
            'order_labor': 0.25,
            'max_num_orders': 5,
            'order_duration': 50
        }),
        ('Gather', {
            'move_labor': 1,
            'collect_labor': 1,
            'skill_dist': 'pareto'
        }),
        # Regional tax components — disabled in Phase 1
        ("RegionalPeriodicBracketTax", {
            "region": "top",
            "planner_id": "p_top",
            "period": 100,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": True,
        }),
        ("RegionalPeriodicBracketTax", {
            "region": "bottom",
            "planner_id": "p_bottom",
            "period": 100,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": True,
        }),
    ],

    "env_layout_file": "map_100x50_water_gaps_3percent_resources.txt",
    "world_size": [100, 50],
    "episode_length": 1000,

    "starting_agent_coin": 10,
    "fixed_four_skill_and_loc": False,

    "n_agents": 4,
    "planner_subclasses": ["TopPlanner", "BottomPlanner"],

    "multi_action_mode_planner": True,
    "multi_action_mode_agents": True,

    "flatten_observations": True,
    "flatten_masks": True,
    "dense_log_frequency": 0 #prev 1
}

# Build wrapper once to capture spaces for policies
env_obj_phase1 = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase1}, verbose=False)

# Per-policy learning rates and spaces
obs_space_a   = env_obj_phase1.observation_space
act_space_a   = env_obj_phase1.action_space
obs_space_top = env_obj_phase1.observation_space_pl["p_top"]
act_space_top = env_obj_phase1.action_space_pl["p_top"]
obs_space_bot = env_obj_phase1.observation_space_pl["p_bottom"]
act_space_bot = env_obj_phase1.action_space_pl["p_bottom"]

policies = {
    "a": (
        None, obs_space_a,   act_space_a,   {"lr": 3e-4}  # agents (Phase-1)
    ),
    "p_top": (
        None, obs_space_top, act_space_top, {"lr": 1e-4}  # planners (Phase-2)
    ),
    "p_bottom": (
        None, obs_space_bot, act_space_bot, {"lr": 1e-4}
    ),
}

def policy_mapping_fun(agent_id):
    aid = str(agent_id)
    if aid.isdigit():
        return "a"
    if aid == "p_top":
        return "p_top"
    if aid == "p_bottom":
        return "p_bottom"
    return "a"

trainer_config_phase1 = {
    "env": RLlibEnvWrapper,
    "env_config": {
        "env_config_dict": env_config_dict_phase1,
        "num_envs_per_worker": 1,   # REQUIRED by your env_wrapper
    },
    "multiagent": {
        "policies": policies,
        "policies_to_train": ["a"],  # agents only
        "policy_mapping_fn": policy_mapping_fun,
    },
    "num_workers": 0,
    "num_envs_per_worker": 1,
    "framework": "tf",               # use TensorFlow (no torch needed)
    "train_batch_size": 4000,
    "sgd_minibatch_size": 4000,
    "num_sgd_iter": 1,
}

trainer_phase1 = PPOTrainer(
    config=trainer_config_phase1,
    logger_creator=make_logger_creator(PHASE1_DIR)
)

# -------- Phase-1: quick test counts (adjust freely) --------
PHASE1_ITERS   = 100   # e.g., quick smoke test; use 2000+ for longer training
PRINT_EVERY_1  = 20
SAVE_EVERY_1   = 100   # save at the end
VERIFY_EVERY_1 = None  # no annealing verify in phase 1

ckpt_phase1_path = train_loop(
    trainer_phase1,
    train_iters=PHASE1_ITERS,
    phase_tag="PHASE 1",
    print_every=PRINT_EVERY_1,
    save_every=SAVE_EVERY_1,
    save_dir=PHASE1_DIR,
    verify_every=VERIFY_EVERY_1,
    verify_fn=None
)
print(f"[PHASE 1] Final checkpoint path: {ckpt_phase1_path}")

# Clean handoff to Phase-2
trainer_phase1.stop()
ray.shutdown()
ray.init(ignore_reinit_error=True, log_to_driver=False)


# =====================================================
# PHASE 2 — ENVIRONMENT (planners-only; taxes ON)
# =====================================================
env_config_dict_phase2 = {
    "scenario_name": "custom/splitworld_overlay_regional",
    "components": [
        ('Build', {
            'skill_dist': 'pareto',
            'payment_max_skill_multiplier': 3,
            'build_labor': 10,
            'payment': 10
        }),
        ('ContinuousDoubleAuction', {
            'max_bid_ask': 10,
            'order_labor': 0.25,
            'max_num_orders': 5,
            'order_duration': 50
        }),
        ('Gather', {
            'move_labor': 1,
            'collect_labor': 1,
            'skill_dist': 'pareto'
        }),
        # Regional tax components — ENABLED in Phase 2 (annealing ON)
        ("RegionalPeriodicBracketTax", {
            "region": "top",
            "planner_id": "p_top",
            "period": 100,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": False,
            "tax_model": "model_wrapper",
            "tax_annealing_schedule": [-100, 0.001],
        }),
        ("RegionalPeriodicBracketTax", {
            "region": "bottom",
            "planner_id": "p_bottom",
            "period": 100,
            "bracket_spacing": "us-federal",
            "usd_scaling": 1000,
            "disable_taxes": False,
            "tax_model": "model_wrapper",
            "tax_annealing_schedule": [-100, 0.001],
        }),
    ],

    "env_layout_file": "map_100x50_water_gaps_3percent_resources.txt",
    "world_size": [100, 50],
    "episode_length": 1000,

    "starting_agent_coin": 10,
    "fixed_four_skill_and_loc": False,

    "n_agents": 4,
    "planner_subclasses": ["TopPlanner", "BottomPlanner"],

    "multi_action_mode_planner": True,
    "multi_action_mode_agents": True,

    "flatten_observations": True,
    "flatten_masks": True,
    "dense_log_frequency": 0 #prev 1
}

# Optional: sanity check spaces (not strictly required)
_ = RLlibEnvWrapper({"env_config_dict": env_config_dict_phase2}, verbose=False)

# NOTE: default is planners-only in Phase-2.
# If you want to also keep agents learning (closer to official docs), set:
TRAIN_AGENTS_IN_PHASE2 = False
policies_to_train_phase2 = ["p_top", "p_bottom"]
if TRAIN_AGENTS_IN_PHASE2:
    policies_to_train_phase2 = ["a", "p_top", "p_bottom"]
    # Optionally reduce agent LR during Phase-2:
    policies["a"][3]["lr"] = 1e-5  # gentler updates for agents

trainer_config_phase2 = {
    "env": RLlibEnvWrapper,
    "env_config": {
        "env_config_dict": env_config_dict_phase2,
        "num_envs_per_worker": 1,   # REQUIRED by your env_wrapper
    },
    "multiagent": {
        "policies": policies,
        "policies_to_train": policies_to_train_phase2,
        "policy_mapping_fn": policy_mapping_fun,
    },
    "num_workers": 0,
    "num_envs_per_worker": 1,
    "framework": "tf",
    "train_batch_size": 4000,
    "sgd_minibatch_size": 4000,
    "num_sgd_iter": 1,
}

trainer_phase2 = PPOTrainer(
    config=trainer_config_phase2,
    logger_creator=make_logger_creator(PHASE2_DIR)
)

# Restore agents from Phase-1
trainer_phase2.restore(ckpt_phase1_path)
print(f"[PHASE 2] Restored from Phase-1 checkpoint: {ckpt_phase1_path}")

# Verify annealing right after restore
verify_annealing(trainer_phase2, tag="[Verify-Start]")

# -------- Phase-2: quick test counts (adjust freely) --------
PHASE2_ITERS   = 250   # e.g., quick smoke test; use 5000+ for longer training
PRINT_EVERY_2  = 25
SAVE_EVERY_2   = 125
VERIFY_EVERY_2 = 125   # check annealing mid-way

_ = train_loop(
    trainer_phase2,
    train_iters=PHASE2_ITERS,
    phase_tag="PHASE 2",
    print_every=PRINT_EVERY_2,
    save_every=SAVE_EVERY_2,
    save_dir=PHASE2_DIR,
    verify_every=VERIFY_EVERY_2,
    verify_fn=verify_annealing
)


2026-03-17 16:02:15,300	INFO resource_spec.py:212 -- Starting Ray with 1.56 GiB memory available for workers and up to 0.8 GiB for objects. You can adjust these settings with ray.init(memory=<bytes>, object_store_memory=<bytes>).


2026-03-17 16:02:15,852	INFO services.py:1165 -- View the Ray dashboard at localhost:8265


: 

In [ ]:
import os

# Limit BLAS thread pools to avoid oversubscription
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

# Limit TensorFlow thread pools
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

# Optional: reduce logging noise
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"


In [ ]:
import ray, psutil
ray.shutdown()  # always start fresh

total = psutil.virtual_memory().total
# Keep plenty of headroom; tweak ratios if you have lots of RAM
ray.init(
    ignore_reinit_error=True,
    log_to_driver=False,
    memory=int(total * 0.60),            # worker heap
    object_store_memory=int(total * 0.20) # plasma store
)

2026-03-17 17:09:19,844	INFO resource_spec.py:212 -- Starting Ray with 9.52 GiB memory available for workers and up to 3.19 GiB for objects. You can adjust these settings with ray.init(memory=<bytes>, object_store_memory=<bytes>).
2026-03-17 17:09:20,376	INFO services.py:1165 -- View the Ray dashboard at localhost:8265


: 

In [ ]:
trainer_config_phase1.update({
    "framework": "tf",
    "train_batch_size": 2000,
    "sgd_minibatch_size": 1000,
    "num_sgd_iter": 1,
    "batch_mode": "truncate_episodes",
    "compress_observations": True,   # reduce data on the wire
})
trainer_config_phase2.update({
    "framework": "tf",
    "train_batch_size": 2000,
    "sgd_minibatch_size": 1000,
    "num_sgd_iter": 1,
    "batch_mode": "truncate_episodes",
    "compress_observations": True,
})

In [ ]:
RESTART_EVERY = 500  # try 500–1000

def segmented_train(trainer_ctor, config, logger, total_iters, phase_tag, save_dir):
    import ray
    iters_done = 0
    last_ckpt = None
    while iters_done < total_iters:
        block = min(RESTART_EVERY, total_iters - iters_done)
        trainer = trainer_ctor(config=config, logger_creator=logger)
        if last_ckpt:
            trainer.restore(last_ckpt)

        # Your existing train_loop call:
        last_ckpt = train_loop(
            trainer,
            train_iters=block,
            phase_tag=phase_tag,
            print_every=25,
            save_every=block,  # save at end of each block
            save_dir=save_dir,
            verify_every=None,
            verify_fn=None
        )

        trainer.stop()
        ray.shutdown()                                  # fully reset
        ray.init(ignore_reinit_error=True, log_to_driver=False,
                 memory=int(total * 0.60), object_store_memory=int(total * 0.20))

        iters_done += block
    return last_ckpt